# 🍜 Scan-Food — Data Visualization

Khảo sát 2 bộ dữ liệu trong `data/recipes` (bỏ qua `nutrition_table.csv`):
- **`images/`** — 1 250 ảnh món ăn
- **`dish_table.csv`** — bảng món ăn với phân loại nhóm

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR   = Path("../data/recipes")
IMAGES_DIR = DATA_DIR / "images"
DISH_CSV   = DATA_DIR / "dish_table.csv"

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Images dir exists:", IMAGES_DIR.exists())
print("Dish CSV exists  :", DISH_CSV.exists())

---
## 1. 🖼️ Images dataset — 9 ảnh ngẫu nhiên

Mỗi ảnh được đặt tên theo `stt` (thứ tự) trong `dish_table.csv`.

In [ ]:
from PIL import Image

df = pd.read_csv(DISH_CSV)
stt_to_name = {
    row["stt"]: row["món ăn"].split("|")[0].strip()
    for _, row in df.iterrows()
}

all_images = sorted(IMAGES_DIR.glob("*.png"), key=lambda p: int(p.stem))
print(f"Tổng số ảnh: {len(all_images)}")

random.seed(42)
sample_imgs = random.sample(all_images, 9)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("9 ảnh ngẫu nhiên — images/", fontsize=15, fontweight="bold", y=1.01)

for ax, img_path in zip(axes.flat, sample_imgs, strict=False):
    # PIL.Image.open xử lý được cả PNG lẫn WEBP giả .png
    img = np.array(Image.open(img_path).convert("RGB"))
    ax.imshow(img)
    ax.axis("off")
    stt       = int(img_path.stem)
    dish_name = stt_to_name.get(stt, img_path.stem)
    if len(dish_name) > 32:
        dish_name = dish_name[:30] + "…"
    ax.set_title(f"#{stt}  {dish_name}", fontsize=8.5, pad=5)

plt.tight_layout()
plt.show()

---
## 2. 🍽️ Dish table dataset — Phân bố nhóm

`dish_table.csv` có **1 250 món** thuộc **19 nhóm**. Dưới đây là tổng quan và các biểu đồ phân bố.

In [ ]:
df = pd.read_csv(DISH_CSV)
print("Shape:", df.shape)
print("\nCác cột:", df.columns.tolist())
df.head(5)

In [ ]:
# Tách phần tiếng Việt (trước '|') cho cả nhóm lẫn tên món
df["nhóm_vn"] = df["nhóm"].str.split("|").str[0].str.strip()
df["tên_vn"]  = df["món ăn"].str.split("|").str[0].str.strip()

def short(s, n=42):
    return s if len(s) <= n else s[:n-1] + "…"

counts_asc  = df["nhóm_vn"].value_counts().sort_values(ascending=True)
counts_desc = df["nhóm_vn"].value_counts()

print(f"Số nhóm: {len(counts_desc)}")
print(f"Tổng món: {len(df)}")
counts_desc

### 2a. Biểu đồ cột ngang — tất cả 19 nhóm

In [ ]:
labels = [short(l) for l in counts_asc.index]
norm   = plt.Normalize(counts_asc.min(), counts_asc.max())
colors = plt.cm.RdYlGn(norm(counts_asc.values))

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(labels, counts_asc.values, color=colors, height=0.72, edgecolor="white")

for bar, val in zip(bars, counts_asc.values, strict=False):
    ax.text(val + 2, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=8, color="#444")

ax.set_xlabel("Số lượng món ăn", fontsize=10)
ax.set_title("Phân bố theo nhóm món ăn (tất cả nhóm)", fontsize=13, fontweight="bold", pad=12)
ax.tick_params(axis="y", labelsize=8)
ax.set_xlim(0, counts_asc.max() * 1.14)
ax.grid(axis="x", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

### 2b. Pie chart — Top 10 nhóm + nhóm khác

In [ ]:
top10  = counts_desc.head(10)
others = counts_desc.iloc[10:].sum()

pie_vals   = list(top10.values) + [others]
pie_labels = [short(l, 35) for l in top10.index] + ["Nhóm khác"]

explode         = [0.03] * 11
explode[0]      = 0.09          # nổi nhóm lớn nhất
colors_pie      = list(plt.cm.tab20.colors[:10]) + ["#bdbdbd"]

fig, ax = plt.subplots(figsize=(10, 7))
wedges, _, autotexts = ax.pie(
    pie_vals, labels=None, autopct="%1.1f%%",
    startangle=140, explode=explode,
    colors=colors_pie, pctdistance=0.82,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
for t in autotexts:
    t.set_fontsize(7.5)
    t.set_color("white")
    t.set_fontweight("bold")

ax.legend(
    wedges, [f"{l}  ({v})" for l, v in zip(pie_labels, pie_vals, strict=False)],
    loc="center left", bbox_to_anchor=(1.02, 0.5),
    fontsize=8, frameon=False,
)
ax.set_title("Top 10 nhóm món ăn", fontsize=13, fontweight="bold", pad=14)
plt.tight_layout()
plt.show()

### 2c. Phân bố độ dài tên món — histogram & boxplot theo top 5 nhóm

In [ ]:
df["name_len"] = df["tên_vn"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# — Histogram tổng thể —
ax = axes[0]
ax.hist(df["name_len"], bins=30, color="#c8564a", edgecolor="white", alpha=0.85)
ax.axvline(df["name_len"].mean(), color="#31302e", linestyle="--", linewidth=1.4,
           label=f"Mean = {df['name_len'].mean():.1f}")
ax.axvline(df["name_len"].median(), color="#4f9678", linestyle=":", linewidth=1.4,
           label=f"Median = {df['name_len'].median():.0f}")
ax.set_xlabel("Số ký tự tên món", fontsize=10)
ax.set_ylabel("Số lượng", fontsize=10)
ax.set_title("Phân bố độ dài tên món ăn", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)

# — Boxplot theo top 5 nhóm —
ax2  = axes[1]
top5 = counts_desc.head(5).index.tolist()
data_box    = [df.loc[df["nhóm_vn"] == g, "name_len"].values for g in top5]
box_palette = ["#c8564a", "#4f9678", "#0075de", "#d4874a", "#7c4daa"]

bp = ax2.boxplot(data_box, patch_artist=True, notch=False,
                 medianprops=dict(color="#31302e", linewidth=2))
for patch, color in zip(bp["boxes"], box_palette, strict=False):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax2.set_xticklabels([short(g, 22) for g in top5], rotation=22, ha="right", fontsize=7.5)
ax2.set_ylabel("Số ký tự tên món", fontsize=10)
ax2.set_title("Độ dài tên — Top 5 nhóm", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nThống kê độ dài tên món:")
df["name_len"].describe().round(1)

---
## 3. 🔧 Chuẩn hoá ảnh — xuất lại toàn bộ thành PNG hợp lệ

Bộ dữ liệu có **726 file WEBP** đặt tên `.png`. Cell dưới đây đọc từng file bằng PIL (nhận dạng đúng format qua magic bytes), chuyển sang RGB và ghi lại thành PNG thực sự — ghi đè tại chỗ.

In [ ]:
from pathlib import Path

from PIL import Image


def detect_format(path: Path) -> str:
    with open(path, "rb") as f:
        h = f.read(12)
    if h[:8] == b"\x89PNG\r\n\x1a\n":
        return "PNG"
    if h[:3] == b"\xff\xd8\xff":
        return "JPEG"
    if h[:4] == b"RIFF":
        return "WEBP"
    if h[:6] in (b"GIF87a", b"GIF89a"):
        return "GIF"
    return "UNKNOWN"

all_imgs = sorted(IMAGES_DIR.glob("*.png"), key=lambda p: int(p.stem))

converted = 0
skipped   = 0
errors    = []

total = len(all_imgs)
print(f"Bắt đầu chuẩn hoá {total} file...\n")

for i, img_path in enumerate(all_imgs, 1):
    fmt = detect_format(img_path)

    if fmt == "PNG":
        skipped += 1
    else:
        try:
            img = Image.open(img_path).convert("RGB")
            img.save(img_path, format="PNG", optimize=False)
            converted += 1
        except Exception as e:
            errors.append((img_path.name, str(e)))

    # In tiến độ mỗi 100 file
    if i % 100 == 0 or i == total:
        print(f"  [{i:>4}/{total}]  converted={converted}  skipped(already PNG)={skipped}  errors={len(errors)}")

print("\n✅ Hoàn thành!")
print(f"   Đã chuyển đổi : {converted} file")
print(f"   Đã là PNG     : {skipped} file")
if errors:
    print(f"   Lỗi ({len(errors)} file):")
    for name, msg in errors:
        print(f"     {name}: {msg}")

In [ ]:
# Xác nhận sau khi chuyển đổi — toàn bộ phải là PNG
from collections import Counter

fmt_after = Counter(detect_format(p) for p in IMAGES_DIR.glob("*.png"))
print("Format breakdown sau khi chuẩn hoá:")
for fmt, count in fmt_after.most_common():
    status = "✅" if fmt == "PNG" else "❌"
    print(f"  {status} {fmt}: {count} file")